
## Step 1: The Time-Based Split & Target Engineering

### 1.1 The Business Target
Our target is `is_late` (1 = Late, 0 = On Time). We define "Late" as `delivery_delay_days > 0`. 
*Note: In Phase 1, we categorized ">3 days" as "Very Late" (4.57% of orders). For Machine Learning, we expand the target to "Any Delay" (>0 days, ~8% of orders). This gives the model more positive examples to learn from, preventing extreme class imbalance.*

### 1.2  Time-Based Split (No Random Splits)
Olist needs to predict the *future*. If we use a standard random `train_test_split`, the model will train on data from 2018 to "predict" an order from 2016. This is called **Data Leakage**—the model has seen the future and will fail in production.

**Our Solution:** We will split the data chronologically. 
*   **Training Set:** Orders placed before August 15, 2018.
*   **Test Set:** Orders placed on or after August 15, 2018.
This simulates training a model on historical data and deploying it to score incoming live orders.

### 1.3 The Sanity Check (Grain Verification)
Before splitting, we will assert that `order_id` is 100% unique. If the row count does not equal the unique order count, we abort. This guarantees the "item-grain" bug from Phase 2 cannot corrupt the Machine Learning model.



In [34]:
    import pandas as pd
    import numpy as np
    from sklearn.metrics import precision_recall_curve, auc
    import matplotlib.pyplot as plt


    # 1. Load the clean data
    df = pd.read_csv('olist_master_clean.csv')

    #  FEATURE ENGINEERING: Target Creation

    df["is_late"] = (df["delivery_delay_days"]> 0).astype(int)
    df["order_purchase_timestamp"] = pd.to_datetime(df['order_purchase_timestamp'])

    # Drop rows with missing timestamps (cannot be used for time split)
    df = df.dropna(subset=['order_purchase_timestamp'])

    #  Time-Based Train/Test Split
    print(df['order_purchase_timestamp'].dt.to_period('Y').value_counts().sort_index())

    # Define the split date 
    split_date = df["order_purchase_timestamp"].quantile(0.80)
    train_df = df[df['order_purchase_timestamp']< split_date].copy()
    test_df = df[df['order_purchase_timestamp']>= split_date].copy()

    print(f"training set: {train_df["order_id"].nunique()} orders before {split_date}")
    print(f"test set: {test_df["order_id"].nunique()} orders after {split_date}")

    # Verify the class imbalance in both sets
    print("\nTarget Distribution (Train):")
    print(train_df['is_late'].value_counts(normalize=True)* 100)

    print("\nTarget Distribution (Test):")
    print(test_df['is_late'].value_counts(normalize=True) * 100)

order_purchase_timestamp
2016      264
2017    43098
2018    52462
Freq: Y-DEC, Name: count, dtype: int64
training set: 76659 orders before 2018-05-27 11:24:51.200000
test set: 19165 orders after 2018-05-27 11:24:51.200000

Target Distribution (Train):
is_late
0    92.533166
1     7.466834
Name: proportion, dtype: float64

Target Distribution (Test):
is_late
0    96.571876
1     3.428124
Name: proportion, dtype: float64



## Step 2: Leakage-Safe Feature Engineering
To simulate a real-world checkout, we can only use data available *at the moment of purchase*. Furthermore, to prevent data leakage, historical seller statistics (like their past late rate) are calculated strictly on the training set and then mapped to the test set. Unseen sellers in the test set receive the global average (cold-start strategy).

In [35]:
print("Engineering New features...")
for data in [train_df, test_df]:
    data['purchase_month'] = data['order_purchase_timestamp'].dt.month
    data['purchase_dayofweek'] = data['order_purchase_timestamp'].dt.dayofweek


print("Engineering seller history (Train only to prevent leakage)...")
# 1. Calculate seller historical stats ONLY on the training data
seller_history = train_df.groupby("seller_id").agg(seller_past_late_rate= ("is_late", "mean"), seller_past_order_count = ("order_id","nunique")).reset_index()

# Calculate the global average late rate (for new sellers)
global_late_rate = train_df['is_late'].mean()
print(f"Global Late Rate (for new sellers): {global_late_rate:.4f}")


# 2. add historical stats to the Train set
train_df = train_df.merge(seller_history, on='seller_id', how='left')

# 3. Map these historical stats to the Test set (unseen sellers become NaN)
test_df = test_df.merge(seller_history, on='seller_id', how='left')


# 4. Cold-start strategy: Fill NaNs in test set with global averages
test_df['seller_past_late_rate'] = test_df['seller_past_late_rate'].fillna(global_late_rate)
test_df['seller_past_order_count'] = test_df['seller_past_order_count'].fillna(0) # 0 past orders


# 5. Frequency Encoding for States (XGBoost prefers numbers over strings)
# Calculate frequency of each state in the training set
state_freq = train_df['seller_state'].value_counts(normalize=True).to_dict()

train_df['seller_state_freq'] = train_df['seller_state'].map(state_freq)
test_df['seller_state_freq'] = test_df['seller_state'].map(state_freq).fillna(0) # Unseen states = 0

cust_state_freq = train_df['customer_state'].value_counts(normalize=True).to_dict()
train_df['cust_state_freq'] = train_df['customer_state'].map(cust_state_freq)
test_df['cust_state_freq'] = test_df['customer_state'].map(cust_state_freq).fillna(0)


Engineering New features...
Engineering seller history (Train only to prevent leakage)...
Global Late Rate (for new sellers): 0.0747


In [36]:
import xgboost as xgb
from sklearn.metrics import classification_report, precision_recall_curve, auc

# ---------------------------------------------------------
# STEP 3: XGBOOST TRAINING (DEFENSIVE MODE)
# ---------------------------------------------------------

# 1. Explicit Feature Selection (FIXED: Added 'review_score' to drop list)
drop_cols = ['order_id', 'customer_id', 'seller_id', 'seller_state', 
             'customer_state', 'order_purchase_timestamp', 
             'order_delivered_customer_date', 'order_estimated_delivery_date',
             'delivery_delay_days', 'delay_status', 'is_late', 'order_month',
             'review_score'] # <-- LEAKAGE FIXED

feature_cols = [c for c in train_df.columns if c not in drop_cols]

X_train = train_df[feature_cols]
y_train = train_df['is_late']

X_test = test_df[feature_cols]
y_test = test_df['is_late']

print(f"Features used by model ({len(feature_cols)}): {feature_cols}")

# 2. Re-calculate scale_pos_weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count

# 3. Initialize XGBoost Classifier
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,           
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)

print("\nTraining XGBoost Model (No Leakage)...")
model.fit(X_train, y_train)

# 4. Predictions & Evaluation
y_pred_probs = model.predict_proba(X_test)[:, 1]
y_pred_class = model.predict(X_test)

precision, recall, _ = precision_recall_curve(y_test, y_pred_probs)
pr_auc = auc(recall, precision)

print(f"\n--- Model Evaluation (Fixed) ---")
print(f"PR-AUC Score: {pr_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_class))

Features used by model (9): ['price', 'freight_value', 'n_items', 'purchase_month', 'purchase_dayofweek', 'seller_past_late_rate', 'seller_past_order_count', 'seller_state_freq', 'cust_state_freq']

Training XGBoost Model (No Leakage)...

--- Model Evaluation (Fixed) ---
PR-AUC Score: 0.0407

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.77      0.86     18508
           1       0.04      0.28      0.07       657

    accuracy                           0.76     19165
   macro avg       0.51      0.53      0.47     19165
weighted avg       0.94      0.76      0.83     19165

